In [3]:
!pip install ipython-sql sqlalchemy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.2/323.2 kB 27.0 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Uninstalling wcwidth-0.2.5:
      Successfully uninstalled wcwidth-0.2.5


In [6]:
import pandas as pd
import sqlite3

# --- Load CSVs into DataFrames ---
users = pd.read_csv("users.csv", parse_dates=[
    "created_at", "plan_started_at", "plan_canceled_at",
    "trial_started_at", "trial_ended_at"
])

events = pd.read_csv("events.csv", parse_dates=["event_time"])

# --- Push into a SQLite file ---
con = sqlite3.connect("analytics.db")
users.to_sql("users", con, index=False, if_exists="replace")
events.to_sql("events", con, index=False, if_exists="replace")

# --- Simple sql() helper, no extra install needed ---
def sql(query):
    return pd.read_sql(query, con)

In [8]:
sql("select * from events limit 100")

,event_name,event_time,user_id
0,ran_ai_scan,2025-08-01 03:51:09.057364,1753
1,ran_ai_scan,2025-08-01 04:17:56.231451,4193
2,ran_ai_scan,2025-08-01 10:27:44.206881,3317
3,ran_ai_scan,2025-08-01 10:41:42.355282,194
4,ran_ai_scan,2025-08-01 10:57:29.004608,3310
...,...,...,...
95,ran_ai_scan,2025-08-02 20:24:35.858823,2623
96,ran_ai_scan,2025-08-02 20:42:09.513586,3167
97,ran_ai_scan,2025-08-02 21:10:01.900196,306
98,ran_ai_scan,2025-08-02 21:28:41.698058,574


In [9]:
sql("select * from users limit 100")

,user_id,user_type,hdyhau,plan_name,created_at,plan_started_at,plan_canceled_at,trial_started_at,trial_ended_at,converted_from_trial
0,1,teacher,search,free,2025-08-14 14:31:31.574153,None,None,None,None,NaN
1,2,professional,search,free,2025-08-31 05:51:45.931002,None,None,None,None,NaN
2,3,teacher,chatgpt,free,2025-08-12 11:52:25.057307,None,None,None,None,NaN
3,4,student,friend,premium,2025-08-26 12:08:03.882223,2025-08-26 12:32:44.543606,None,None,None,0.0
4,5,teacher,search,free,2025-08-02 23:28:48.074932,None,None,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...
95,96,student,search,free,2025-08-30 00:29:16.404608,None,None,None,None,NaN
96,97,professional,search,free,2025-08-31 14:03:12.943294,None,None,None,None,NaN
97,98,teacher,search,free,2025-08-04 16:08:52.902740,None,None,None,None,NaN
98,99,student,social_media,free,2025-08-19 05:39:49.638860,None,None,None,None,NaN


In [10]:
sql("select * from users where converted_from_trial = True limit 100")

,user_id,user_type,hdyhau,plan_name,created_at,plan_started_at,plan_canceled_at,trial_started_at,trial_ended_at,converted_from_trial
0,23,student,friend,free,2025-08-14 19:56:24.522341,2025-09-09 00:29:41.315448,2025-09-10 16:53:57.193149,2025-09-02 18:29:40.789524,2025-09-09 18:29:40.789524,1
1,31,student,friend,free,2025-08-02 06:57:39.361658,2025-09-19 02:35:21.077539,2025-09-19 20:24:18.972631,2025-09-12 09:44:33.958662,2025-09-19 09:44:33.958662,1
2,54,student,search,premium,2025-08-23 04:15:45.740556,2025-09-07 15:50:48.837217,None,2025-09-01 13:27:30.102845,2025-09-08 13:27:30.102845,1
3,103,teacher,chatgpt,premium,2025-08-04 07:10:33.967745,2025-09-15 23:46:19.692541,None,2025-09-09 18:54:29.029091,2025-09-16 18:54:29.029091,1
4,109,professional,social_media,free,2025-08-24 22:26:54.364053,2025-09-10 03:23:17.374438,2025-09-11 03:28:55.326130,2025-09-03 23:34:04.543885,2025-09-10 23:34:04.543885,1
...,...,...,...,...,...,...,...,...,...,...
95,6379,student,social_media,premium,2025-09-30 00:42:15.862336,2025-10-06 04:56:53.896316,None,2025-09-30 01:28:46.719720,2025-10-07 01:28:46.719720,1
96,6424,teacher,search,premium,2025-09-06 16:31:47.618861,2025-09-12 00:12:30.637670,None,2025-09-06 17:12:46.079336,2025-09-13 17:12:46.079336,1
97,6426,teacher,search,premium,2025-09-15 01:41:05.530688,2025-09-20 03:42:08.098704,None,2025-09-15 01:55:12.757461,2025-09-22 01:55:12.757461,1
98,6550,student,friend,premium,2025-09-30 03:37:33.148869,2025-10-06 13:27:39.174284,None,2025-09-30 03:57:46.403885,2025-10-07 03:57:46.403885,1


In [12]:
sql("select converted_from_trial from users group by 1")

,converted_from_trial
0,NaN
1,0.0
2,1.0


In [15]:
sql("select event_name from events group by 1")

,event_name,count(distinct user_id)
0,in_experiment_group_freemium,8519
1,in_experiment_group_paid_only,8054
2,ran_ai_scan,12818
3,ran_writing_feedback_scan,1349
4,started_free_trial,1713
5,started_paid_plan,799
6,successfully_referred_friend,551
7,viewed_document_history,1521
8,viewed_out_of_credits_modal,935


In [18]:
print(sqlite3.sqlite_version)

3.41.2


In [47]:
sql("""
WITH control_users AS (
    SELECT u.user_id
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_freemium'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
treatment_users AS (
    SELECT u.user_id
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_paid_only'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
control_post AS (
    SELECT e.user_id, e.event_name, e.event_time
    FROM events e
    JOIN control_users c ON e.user_id = c.user_id
    WHERE e.event_name != 'in_experiment_group_freemium'
),
treatment_post AS (
    SELECT e.user_id, e.event_name, e.event_time
    FROM events e
    JOIN treatment_users t ON e.user_id = t.user_id
    WHERE e.event_name != 'in_experiment_group_paid_only'
),
control_first AS (
    SELECT user_id, event_name
    FROM (
        SELECT user_id, event_name,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM control_post
    ) WHERE rn = 1
),
treatment_first AS (
    SELECT user_id, event_name
    FROM (
        SELECT user_id, event_name,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM treatment_post
    ) WHERE rn = 1
),
n AS (
    SELECT
        (SELECT COUNT(*) FROM control_users)   AS n_ctrl,
        (SELECT COUNT(*) FROM treatment_users) AS n_trt
)

SELECT 'Total users' AS event,
    n_ctrl AS control_n,
    n_trt  AS treatment_n,
    NULL   AS control_pct,
    NULL   AS treatment_pct
FROM n

UNION ALL
SELECT event_name,
    SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END) AS control_n,
    SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END) AS treatment_n,
    ROUND(100.0 * SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END) / (SELECT n_ctrl FROM n), 2),
    ROUND(100.0 * SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END) / (SELECT n_trt  FROM n), 2)
FROM (
    SELECT event_name, 'control'   AS grp FROM control_first
    UNION ALL
    SELECT event_name, 'treatment' AS grp FROM treatment_first
)
WHERE event_name IN ('ran_ai_scan','started_free_trial','ran_writing_feedback_scan','started_paid_plan')
GROUP BY event_name

UNION ALL
SELECT 'no_events_after_assignment',
    (SELECT n_ctrl FROM n) - (SELECT COUNT(DISTINCT user_id) FROM control_post),
    (SELECT n_trt  FROM n) - (SELECT COUNT(DISTINCT user_id) FROM treatment_post),
    ROUND(100.0 * ((SELECT n_ctrl FROM n) - (SELECT COUNT(DISTINCT user_id) FROM control_post))   / (SELECT n_ctrl FROM n), 2),
    ROUND(100.0 * ((SELECT n_trt  FROM n) - (SELECT COUNT(DISTINCT user_id) FROM treatment_post)) / (SELECT n_trt  FROM n), 2)

UNION ALL
SELECT 'ever_started_paid_plan',
    (SELECT COUNT(DISTINCT user_id) FROM control_post   WHERE event_name = 'started_paid_plan'),
    (SELECT COUNT(DISTINCT user_id) FROM treatment_post WHERE event_name = 'started_paid_plan'),
    ROUND(100.0 * (SELECT COUNT(DISTINCT user_id) FROM control_post   WHERE event_name = 'started_paid_plan') / (SELECT n_ctrl FROM n), 2),
    ROUND(100.0 * (SELECT COUNT(DISTINCT user_id) FROM treatment_post WHERE event_name = 'started_paid_plan') / (SELECT n_trt  FROM n), 2)

UNION ALL
SELECT 'ever_viewed_out_of_credits_modal',
    (SELECT COUNT(DISTINCT user_id) FROM control_post   WHERE event_name = 'viewed_out_of_credits_modal'),
    (SELECT COUNT(DISTINCT user_id) FROM treatment_post WHERE event_name = 'viewed_out_of_credits_modal'),
    ROUND(100.0 * (SELECT COUNT(DISTINCT user_id) FROM control_post   WHERE event_name = 'viewed_out_of_credits_modal') / (SELECT n_ctrl FROM n), 2),
    ROUND(100.0 * (SELECT COUNT(DISTINCT user_id) FROM treatment_post WHERE event_name = 'viewed_out_of_credits_modal') / (SELECT n_trt  FROM n), 2)
""")

,event,control_n,treatment_n,control_pct,treatment_pct
0,Total users,6400,5889,NaN,NaN
1,ran_ai_scan,5911,1319,92.36,22.40
2,ran_writing_feedback_scan,134,0,2.09,0.00
3,started_free_trial,0,1267,0.00,21.51
4,started_paid_plan,53,175,0.83,2.97
5,no_events_after_assignment,302,3128,4.72,53.12
6,ever_started_paid_plan,255,294,3.98,4.99
7,ever_viewed_out_of_credits_modal,566,0,8.84,0.00


In [43]:
from statsmodels.stats.proportion import proportions_ztest

# counts of successes
count = [255, 294]

# total samples
nobs = [6400, 5889]

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print("z-stat:", z_stat)
print("p-value:", p_value)

z-stat: -2.702105470466466
p-value: 0.006890190190754782


In [41]:
sql("""WITH control_users AS (
    SELECT u.user_id, u.hdyhau
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_freemium'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
treatment_users AS (
    SELECT u.user_id, u.hdyhau
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_paid_only'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
control_post AS (
    SELECT e.user_id, e.event_name, e.event_time, c.hdyhau
    FROM events e
    JOIN control_users c ON e.user_id = c.user_id
    WHERE e.event_name != 'in_experiment_group_freemium'
),
treatment_post AS (
    SELECT e.user_id, e.event_name, e.event_time, t.hdyhau
    FROM events e
    JOIN treatment_users t ON e.user_id = t.user_id
    WHERE e.event_name != 'in_experiment_group_paid_only'
),
control_first AS (
    SELECT user_id, event_name, hdyhau
    FROM (
        SELECT user_id, event_name, hdyhau,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM control_post
    ) WHERE rn = 1
),
treatment_first AS (
    SELECT user_id, event_name, hdyhau
    FROM (
        SELECT user_id, event_name, hdyhau,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM treatment_post
    ) WHERE rn = 1
),
n_ctrl AS (SELECT hdyhau, COUNT(*) AS n_ctrl FROM control_users   GROUP BY hdyhau),
n_trt  AS (SELECT hdyhau, COUNT(*) AS n_trt  FROM treatment_users GROUP BY hdyhau),
n AS (
    SELECT COALESCE(c.hdyhau, t.hdyhau) AS hdyhau,
           COALESCE(c.n_ctrl, 0) AS n_ctrl,
           COALESCE(t.n_trt,  0) AS n_trt
    FROM n_ctrl c LEFT JOIN n_trt t ON c.hdyhau = t.hdyhau
),
ctrl_active    AS (SELECT hdyhau, COUNT(DISTINCT user_id) AS active FROM control_post   GROUP BY hdyhau),
trt_active     AS (SELECT hdyhau, COUNT(DISTINCT user_id) AS active FROM treatment_post  GROUP BY hdyhau),
ctrl_ever_paid AS (SELECT hdyhau, COUNT(DISTINCT user_id) AS paid FROM control_post   WHERE event_name = 'started_paid_plan' GROUP BY hdyhau),
trt_ever_paid  AS (SELECT hdyhau, COUNT(DISTINCT user_id) AS paid FROM treatment_post WHERE event_name = 'started_paid_plan' GROUP BY hdyhau)

SELECT hdyhau, 'Total users' AS event,
    n_ctrl AS control_n, n_trt AS treatment_n, NULL AS control_pct, NULL AS treatment_pct
FROM n

UNION ALL
SELECT f.hdyhau, f.event_name,
    SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END),
    SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END) / n.n_ctrl, 2),
    ROUND(100.0 * SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END) / n.n_trt,  2)
FROM (
    SELECT event_name, hdyhau, 'control'   AS grp FROM control_first
    UNION ALL
    SELECT event_name, hdyhau, 'treatment' AS grp FROM treatment_first
) f
JOIN n ON f.hdyhau = n.hdyhau
WHERE f.event_name IN ('ran_ai_scan','started_free_trial','ran_writing_feedback_scan','started_paid_plan')
GROUP BY f.hdyhau, f.event_name

UNION ALL
SELECT n.hdyhau, 'no_events_after_assignment',

    n.n_ctrl - COALESCE(ca.active, 0), n.n_trt - COALESCE(ta.active, 0),
    ROUND(100.0 * (n.n_ctrl - COALESCE(ca.active, 0)) / n.n_ctrl, 2),
    ROUND(100.0 * (n.n_trt  - COALESCE(ta.active, 0)) / n.n_trt,  2)
FROM n
LEFT JOIN ctrl_active ca ON n.hdyhau = ca.hdyhau
LEFT JOIN trt_active  ta ON n.hdyhau = ta.hdyhau

UNION ALL
SELECT n.hdyhau, 'ever_started_paid_plan',
    COALESCE(cp.paid, 0), COALESCE(tp.paid, 0),
    ROUND(100.0 * COALESCE(cp.paid, 0) / n.n_ctrl, 2),
    ROUND(100.0 * COALESCE(tp.paid, 0) / n.n_trt,  2)
FROM n
LEFT JOIN ctrl_ever_paid cp ON n.hdyhau = cp.hdyhau
LEFT JOIN trt_ever_paid  tp ON n.hdyhau = tp.hdyhau

ORDER BY hdyhau, event """)

,hdyhau,event,control_n,treatment_n,control_pct,treatment_pct
0,chatgpt,Total users,470,509,NaN,NaN
1,chatgpt,ever_started_paid_plan,25,30,5.32,5.89
2,chatgpt,no_events_after_assignment,23,285,4.89,55.99
3,chatgpt,ran_ai_scan,437,106,92.98,20.83
4,chatgpt,ran_writing_feedback_scan,6,0,1.28,0.00
5,chatgpt,started_free_trial,0,101,0.00,19.84
6,chatgpt,started_paid_plan,4,17,0.85,3.34
7,friend,Total users,3079,2439,NaN,NaN
8,friend,ever_started_paid_plan,94,127,3.05,5.21
9,friend,no_events_after_assignment,130,1319,4.22,54.08


In [44]:

# counts of successes
count = [94, 127]

# total samples
nobs = [3079, 2439]

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print("z-stat:", z_stat)
print("p-value:", p_value)

z-stat: -4.05283489405364
p-value: 5.060071423647729e-05


In [45]:
sql("""WITH control_users AS (
    SELECT u.user_id, u.user_type
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_freemium'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
treatment_users AS (
    SELECT u.user_id, u.user_type
    FROM users u
    JOIN events e ON u.user_id = e.user_id
        AND e.event_name = 'in_experiment_group_paid_only'
    WHERE DATE(u.created_at) >= '2025-09-01'
),
control_post AS (
    SELECT e.user_id, e.event_name, e.event_time, c.user_type
    FROM events e
    JOIN control_users c ON e.user_id = c.user_id
    WHERE e.event_name != 'in_experiment_group_freemium'
),
treatment_post AS (
    SELECT e.user_id, e.event_name, e.event_time, t.user_type
    FROM events e
    JOIN treatment_users t ON e.user_id = t.user_id
    WHERE e.event_name != 'in_experiment_group_paid_only'
),
control_first AS (
    SELECT user_id, event_name, user_type
    FROM (
        SELECT user_id, event_name, user_type,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM control_post
    ) WHERE rn = 1
),
treatment_first AS (
    SELECT user_id, event_name, user_type
    FROM (
        SELECT user_id, event_name, user_type,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
        FROM treatment_post
    ) WHERE rn = 1
),
n_ctrl AS (SELECT user_type, COUNT(*) AS n_ctrl FROM control_users   GROUP BY user_type),
n_trt  AS (SELECT user_type, COUNT(*) AS n_trt  FROM treatment_users GROUP BY user_type),
n AS (
    SELECT COALESCE(c.user_type, t.user_type) AS user_type,
           COALESCE(c.n_ctrl, 0) AS n_ctrl,
           COALESCE(t.n_trt,  0) AS n_trt
    FROM n_ctrl c LEFT JOIN n_trt t ON c.user_type = t.user_type
),
ctrl_active    AS (SELECT user_type, COUNT(DISTINCT user_id) AS active FROM control_post   GROUP BY user_type),
trt_active     AS (SELECT user_type, COUNT(DISTINCT user_id) AS active FROM treatment_post  GROUP BY user_type),
ctrl_ever_paid AS (SELECT user_type, COUNT(DISTINCT user_id) AS paid FROM control_post   WHERE event_name = 'started_paid_plan' GROUP BY user_type),
trt_ever_paid  AS (SELECT user_type, COUNT(DISTINCT user_id) AS paid FROM treatment_post WHERE event_name = 'started_paid_plan' GROUP BY user_type)

SELECT user_type, 'Total users' AS event,
    n_ctrl AS control_n, n_trt AS treatment_n, NULL AS control_pct, NULL AS treatment_pct
FROM n

UNION ALL
SELECT f.user_type, f.event_name,
    SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END),
    SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN grp='control'   THEN 1 ELSE 0 END) / n.n_ctrl, 2),
    ROUND(100.0 * SUM(CASE WHEN grp='treatment' THEN 1 ELSE 0 END) / n.n_trt,  2)
FROM (
    SELECT event_name, user_type, 'control'   AS grp FROM control_first
    UNION ALL
    SELECT event_name, user_type, 'treatment' AS grp FROM treatment_first
) f
JOIN n ON f.user_type = n.user_type
WHERE f.event_name IN ('ran_ai_scan','started_free_trial','ran_writing_feedback_scan','started_paid_plan')
GROUP BY f.user_type, f.event_name

UNION ALL
SELECT n.user_type, 'no_events_after_assignment',
    n.n_ctrl - COALESCE(ca.active, 0), n.n_trt - COALESCE(ta.active, 0),
    ROUND(100.0 * (n.n_ctrl - COALESCE(ca.active, 0)) / n.n_ctrl, 2),
    ROUND(100.0 * (n.n_trt  - COALESCE(ta.active, 0)) / n.n_trt,  2)
FROM n
LEFT JOIN ctrl_active ca ON n.user_type = ca.user_type
LEFT JOIN trt_active  ta ON n.user_type = ta.user_type

UNION ALL
SELECT n.user_type, 'ever_started_paid_plan',
    COALESCE(cp.paid, 0), COALESCE(tp.paid, 0),
    ROUND(100.0 * COALESCE(cp.paid, 0) / n.n_ctrl, 2),
    ROUND(100.0 * COALESCE(tp.paid, 0) / n.n_trt,  2)
FROM n
LEFT JOIN ctrl_ever_paid cp ON n.user_type = cp.user_type
LEFT JOIN trt_ever_paid  tp ON n.user_type = tp.user_type

ORDER BY user_type, event """)

,user_type,event,control_n,treatment_n,control_pct,treatment_pct
0,professional,Total users,1554,1490,NaN,NaN
1,professional,ever_started_paid_plan,64,58,4.12,3.89
2,professional,no_events_after_assignment,76,806,4.89,54.09
3,professional,ran_ai_scan,1441,332,92.73,22.28
4,professional,ran_writing_feedback_scan,24,0,1.54,0.00
5,professional,started_free_trial,0,311,0.00,20.87
6,professional,started_paid_plan,13,41,0.84,2.75
7,student,Total users,3355,3359,NaN,NaN
8,student,ever_started_paid_plan,146,176,4.35,5.24
9,student,no_events_after_assignment,173,1748,5.16,52.04


In [46]:
# counts of successes
count = [45, 60]

# total samples
nobs = [1491, 1040]

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print("z-stat:", z_stat)
print("p-value:", p_value)

z-stat: -3.414848056350893
p-value: 0.0006381760050604549
